# Corpus Caesarianum. Perseus TEI Scraper & Audit Notebook

**Source:** [PerseusDL/canonical-latinLit](https://github.com/PerseusDL/canonical-latinLit) (CC BY-SA 4.0)  
**Output:** `full_data_text.csv`. Pipe-delimited corpus file for the analysis pipeline

This notebook is a step-by-step walkthrough of the `perseus_scraper.py` pipeline. Rather than treating the scraper as a black box, every transformation is exposed here with an explanation of why each decision was made and what effect it has on the data.

**The data model**

The pipe character (`|`) is the CSV delimiter because it almost never appears in Latin prose, so column-splitting is unambiguous. The one place it does appear in the Perseus TEI files is inside `<num>` elements for sestertius amounts (e.g. `|XX̄|`). That case is handled explicitly below.

| Column | Type | Description |
|--------|------|-------------|
| *(index)* | int | 0-based integer row index, continuous across all five texts |
| `commentary` | str | Short label for the text |
| `book` | int | Book number (always `1` for single-book texts) |
| `text` | str | Full Latin prose of the chapter, whitespace-normalised |
| `src` | str | CTS URN for the passage |
| `chapter` | int | 0-indexed chapter number within the book |

The chapter is the unit of analysis. For the three-level texts (BG, BC), all sections within a chapter are concatenated into a single row. For the two-level texts (BAlex, BAf, BH), each chapter div maps directly to a row.

The `chapter` column is 0-indexed (first chapter of a book = `0`). This matches the convention in prior work and matters for array slicing downstream. The `src` CTS URN uses the human-readable chapter number from the XML `n=` attribute, which is 1-based for all five texts.

**The corpus**

The *Corpus Caesarianum* is five works transmitted together in the manuscript tradition, collectively attributed (with varying confidence) to Caesar and members of his entourage.

| Label | Full title | Author (traditional) | Books | Chapters |
|-------|-----------|----------------------|-------|----------|
| `gallic` | *De Bello Gallico* | Caesar (bks 1-7) + Hirtius (bk 8) | 8 | 404 |
| `civil` | *Bellum Civile* | Caesar | 3 | 243 |
| `alexandrine` | *Bellum Alexandrinum* | Hirtius (trad.) | 1 | 78 |
| `african` | *Bellum Africum* | unknown | 1 | 98 |
| `spanish` | *Bellum Hispaniense* | unknown | 1 | 42 |

The attribution questions are what the downstream analysis is designed to investigate. So the labels are agnostic: `gallic` doesn't imply Caesar wrote book 8, `alexandrine` doesn't imply Hirtius wrote the *Bellum Alexandrinum*, and so on.

The Perseus TEI files come in two structural variants, named after the filename suffix:

- **lat2** (`gallic`, `civil`): three-level hierarchy -- book > chapter > section > `<p>`. The `has_books=True` flag adds a book-iteration path for the parser.
- **lat1** (`alexandrine`, `african`, `spanish`): two-level hierarchy -- chapter > `<p>`. The `has_books=False` flag runs a flat chapter-iteration path.

Each row carries a [CTS URN](https://cite-architecture.github.io/ctsurn/), a canonical scholarly identifier for the passage:

```
urn:cts:latinLit:phi0448.phi001.perseus-lat2:1.1
                 phi identifier (author.work)  book.chapter
```

The PHI numbers (`phi0448`, `phi0428`, etc.) are the Perseus/TLG identifiers for the author-work pair. They appear in the repository directory names and the filenames.

**Setup**

You'll need `lxml` installed in your venv (`pip install lxml`). Everything else is standard library.

In [ ]:
import logging
import os
import re
import sys
import unicodedata
import urllib.request
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path
from lxml import etree as lxml_etree

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("cc_notebook")
log.info("Environment ready.")

15:15:44  INFO  Environment ready.


**Corpus configuration**

In [2]:
TEI_NS     = "http://www.tei-c.org/ns/1.0"
GITHUB_RAW = "https://raw.githubusercontent.com/PerseusDL/canonical-latinLit/master/data"

CORPUS = [
    {
        "commentary": "gallic",
        "path":       "phi0448/phi001/phi0448.phi001.perseus-lat2.xml",
        "has_books":  True,
        "cts_base":   "urn:cts:latinLit:phi0448.phi001.perseus-lat2",
    },
    {
        "commentary": "civil",
        "path":       "phi0448/phi002/phi0448.phi002.perseus-lat2.xml",
        "has_books":  True,
        "cts_base":   "urn:cts:latinLit:phi0448.phi002.perseus-lat2",
    },
    {
        "commentary": "alexandrine",
        "path":       "phi0428/phi001/phi0428.phi001.perseus-lat1.xml",
        "has_books":  False,
        "cts_base":   "urn:cts:latinLit:phi0428.phi001.perseus-lat1",
    },
    {
        "commentary": "african",
        "path":       "phi0426/phi001/phi0426.phi001.perseus-lat1.xml",
        "has_books":  False,
        "cts_base":   "urn:cts:latinLit:phi0426.phi001.perseus-lat1",
    },
    {
        "commentary": "spanish",
        "path":       "phi0430/phi001/phi0430.phi001.perseus-lat1.xml",
        "has_books":  False,
        "cts_base":   "urn:cts:latinLit:phi0430.phi001.perseus-lat1",
    },
]

print(f"{'label':>12}  {'books?':>6}  {'PHI path'}")
print("-" * 60)
for entry in CORPUS:
    print(f"{entry['commentary']:>12}  {str(entry['has_books']):>6}  {entry['path']}")

       label  books?  PHI path
------------------------------------------------------------
      gallic    True  phi0448/phi001/phi0448.phi001.perseus-lat2.xml
       civil    True  phi0448/phi002/phi0448.phi002.perseus-lat2.xml
 alexandrine   False  phi0428/phi001/phi0428.phi001.perseus-lat1.xml
     african   False  phi0426/phi001/phi0426.phi001.perseus-lat1.xml
     spanish   False  phi0430/phi001/phi0430.phi001.perseus-lat1.xml


**File paths**

By default this uses per-file overrides -- just point each key at the downloaded XML file. Set a value to `None` if you want the resolver to fall back to `REPO_DIR` for that text.

If you want to use a full repo clone instead, set `REPO_DIR` to the root of a `canonical-latinLit` clone (or its `data/` subdirectory) and set all overrides to `None`:

```bash
git clone --depth 1 https://github.com/PerseusDL/canonical-latinLit.git
```

^ But by default you should not need to worry about this unless you'd like to manage the full repo clone.

In [ ]:
# Per-file paths (primary). Set to None to fall back to REPO_DIR.
FILE_OVERRIDES = {
    "gallic":      None, #"phi0448.phi001.perseus-lat2.xml",
    "civil":       None, #"phi0448.phi002.perseus-lat2.xml",
    "alexandrine": None, #"phi0428.phi001.perseus-lat1.xml",
    "african":     None, #"phi0426.phi001.perseus-lat1.xml",
    "spanish":     None, #"phi0430.phi001.perseus-lat1.xml",
}

# Fallback: root of a canonical-latinLit clone, or its data/ subdir. raw.githubusercontent.com is unreliable for large files, so this is often the best option.
REPO_DIR = "canonical-latinLit" #None

OUTPUT_PATH = "full_data_text_perseus.csv"

In [4]:
def resolve_path(entry, repo_dir, overrides):
    """
    Find the XML file for a corpus entry.

    Resolution order:
      1. Explicit override path (FILE_OVERRIDES)
      2. <repo_dir>/data/<path>  (repo root supplied)
      3. <repo_dir>/<path>       (data/ subdir supplied)
      4. <repo_dir>/<filename>   (flat copy in repo root)
    """
    commentary = entry["commentary"]
    if overrides.get(commentary):
        p = Path(overrides[commentary])
        if p.exists():
            return p
        raise FileNotFoundError(f"Override path not found: {p}")

    if repo_dir is None:
        return None

    repo = Path(repo_dir)
    for candidate in [
        repo / "data" / entry["path"],
        repo / entry["path"],
        repo / Path(entry["path"]).name,
    ]:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"Could not find {entry['path']} under {repo_dir}.\n"
        "Try setting FILE_OVERRIDES or pointing REPO_DIR at the right place."
    )


def download_xml(entry, dest_dir):
    """Download a single XML file from GitHub raw content."""
    url  = f"{GITHUB_RAW}/{entry['path']}"
    name = Path(entry["path"]).name
    dest = Path(dest_dir) / name

    if dest.exists():
        log.info(f"  Already cached: {dest}")
        return dest

    log.info(f"  Fetching {url}")
    req = urllib.request.Request(
        url, headers={"User-Agent": "caesarianum-notebook/1.0"}
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        content = resp.read()
    dest.write_bytes(content)
    log.info(f"  Saved {len(content):,} bytes to {dest}")
    return dest


xml_paths = {}
acquisition_log = []

for entry in CORPUS:
    commentary = entry["commentary"]
    override   = FILE_OVERRIDES.get(commentary)

    try:
        if override:
            p = Path(override)
            if not p.exists():
                log.info(f"  {commentary}: {p} not found locally, downloading...")
                os.makedirs("perseus_xml", exist_ok=True)
                p = download_xml(entry, "perseus_xml")
            method = "override" if Path(override).exists() else "download"
        else:
            p = resolve_path(entry, REPO_DIR, {})
            method = "repo"

        size_kb = p.stat().st_size // 1024
        xml_paths[commentary] = p
        acquisition_log.append((commentary, method, str(p), size_kb))
        log.info(f"  {commentary:>12}  [{method}]  {p}  ({size_kb} KB)")

    except (FileNotFoundError, RuntimeError) as e:
        log.error(f"  {commentary}: {e}")

print(f"\nResolved {len(xml_paths)}/5 files.")

15:15:44  INFO    gallic: phi0448.phi001.perseus-lat2.xml not found locally, downloading...
15:15:44  INFO    Fetching https://raw.githubusercontent.com/PerseusDL/canonical-latinLit/master/data/phi0448/phi001/phi0448.phi001.perseus-lat2.xml


TimeoutError: The read operation timed out

**TEI structure**

Before parsing it's worth looking at what the XML actually looks like in each variant. The lat2 files (BG, BC) have a three-level hierarchy:

```xml
<TEI xmlns="http://www.tei-c.org/ns/1.0">
  <text>
    <body>
      <div type="edition" xml:lang="lat">
        <div type="textpart" subtype="book" n="1">
          <div type="textpart" subtype="chapter" n="1">
            <div type="textpart" subtype="section" n="1">
              <p>Gallia est omnis divisa in partes tres...</p>
            </div>
            <div type="textpart" subtype="section" n="2">
              <p>Eorum una pars...</p>
            </div>
          </div>
        </div>
      </div>
    </body>
  </text>
</TEI>
```

The lat1 files (BAlex, BAf, BH) skip the book level entirely:

```xml
<TEI xmlns="http://www.tei-c.org/ns/1.0">
  <text>
    <body>
      <div type="edition" xml:lang="lat">
        <div type="textpart" subtype="chapter" n="1">
          <p>Bello Alexandrino conflato...</p>
        </div>
      </div>
    </body>
  </text>
</TEI>
```

The parser finds divs by their `type` and `subtype` attributes rather than by position in the tree. This keeps it robust to wrapper elements, minor structural differences between files, and any future changes to the repo. I use the `_iter_divs()` function to recurse through non-textpart wrappers but stops when it hits a textpart of a different subtype -- so a book search won't descend into chapters, and a chapter search won't descend into sections.

The cell below prints the actual div structure of one lat2 and one lat1 file as a sanity check before the full parse.

In [ ]:
def inspect_tei_structure(xml_path, max_depth=4):
    """Print the first few levels of div structure in a TEI file."""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    def _walk(elem, depth=0):
        if depth > max_depth:
            return
        local = elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag
        attrs = {
            k.split("}")[-1]: v
            for k, v in elem.attrib.items()
            if k.split("}")[-1] in ("type", "subtype", "n", "xml:lang", "lang")
        }
        text_preview = ""
        if elem.text and elem.text.strip():
            preview = elem.text.strip()[:40].replace("\n", " ")
            text_preview = f'  "{preview}..."'
        indent = "  " * depth
        attr_str = " ".join(f'{k}={v!r}' for k, v in attrs.items())
        print(f"{indent}<{local} {attr_str}>{text_preview}")

        for child in elem:
            child_local = child.tag.split("}")[-1] if "}" in child.tag else child.tag
            if child_local in ("div", "body", "text", "group"):
                _walk(child, depth + 1)
                siblings = [
                    c for c in elem
                    if (c.tag.split("}")[-1] if "}" in c.tag else c.tag) == child_local
                ]
                if len(siblings) > 1:
                    print(f"{indent}  ... ({len(siblings)-1} more <{child_local}>)")
                break

    _walk(root)


for label in ["gallic", "alexandrine"]:
    if label in xml_paths:
        print(f"\n{'='*60}")
        print(f"Structure of '{label}' ({xml_paths[label].name}):")
        print('='*60)
        inspect_tei_structure(xml_paths[label])

**What gets extracted and what gets dropped**

A TEI file contains a lot more than the Latin prose -- editorial notes, page breaks, conjectural emendations, headers, and so on. For authorship attribution purposes, I want the running Latin prose as the editor believes the author wrote it. Maybe, just maybe, we'd be interested in the other fields down the road. The breakdown below is meant to help understand the other elements. But, for now we skip them.

| Element | Action | Reason |
|---------|--------|---------|
| `<p>` | include | Primary prose container |
| `<add>` | include | Editorial supplement filling a gap |
| `<supplied>` | include | Same logic as `<add>` |
| `<sic>` | include | Error in MS but text is present |
| `<num>` | include | Roman numerals -- part of the author's vocabulary |
| `<foreign>` | include | Foreign words embedded in Latin prose |
| `<hi>` | include | Typographic highlighting, no semantic change |
| `<q>` | include | Quotations |
| `<unclear>` | include | Partially legible but present in the MS |
| `<abbr>` | include | Abbreviations |
| `<note>` | skip | Editorial notes, often in English |
| `<head>` | skip | Section headers (e.g. "COMMENTARIUS PRIMUS") -- editorial |
| `<pb>` | skip | Page-break markers, no textual content |
| `<lb>` | skip | Line-break markers, no textual content |
| `<milestone>` | skip | Reference markers, no textual content |
| `<gap>` | skip | Lacunae -- text is absent from the MS |
| `<del>` | skip | Text the editor believes is spurious/interpolated |
| `<figure>` | skip | Figures |

One thing that trips people up: even for skipped elements, I always keep the `.tail`. In XML, the tail is the text that comes after the closing tag but before the next sibling -- it belongs to the parent's prose flow. So given

```xml
<p>Hoc fecit <note>cf. Cic.</note> Caesar.</p>
```

the `<note>` content (`cf. Cic.`) gets dropped, but its tail (` Caesar.`) has to be kept or the sentence breaks. The demo below shows this in action. This really confounded me for a awhile until I was able to figure this out.

In [ ]:
_SKIP_CONTENT = frozenset({
    "note", "head", "pb", "lb", "milestone",
    "gap", "del", "figure",
})


def _local(tag):
    """Strip the TEI namespace prefix from a tag string."""
    return tag.split("}")[-1] if "}" in tag else tag


def _extract_text(elem, _audit=None):
    """
    Recursively pull Latin text out of a TEI element.
    Skips content of elements in _SKIP_CONTENT but always keeps
    their tail, which belongs to the parent's prose flow.
    If _audit is a list, suppression events get appended to it.
    """
    local = _local(elem.tag)

    if local in _SKIP_CONTENT:
        if _audit is not None and elem.text and elem.text.strip():
            _audit.append({
                "element":    local,
                "suppressed": elem.text.strip()[:80],
                "tail_kept":  (elem.tail or "").strip()[:40],
            })
        return elem.tail or ""

    parts = []
    if elem.text:
        parts.append(elem.text)
    for child in elem:
        parts.append(_extract_text(child, _audit))
    if elem.tail:
        parts.append(elem.tail)
    return "".join(parts)


# Demo
demo_xml = """<p xmlns='http://www.tei-c.org/ns/1.0'>
  Hoc fecit <note>cf. Cic. Fam. 1.1</note> Caesar,
  qui <del>omnino</del> imperium <pb n='p.12'/> tenebat.
</p>"""

demo_elem  = ET.fromstring(demo_xml)
audit_events = []
raw_text   = _extract_text(demo_elem, _audit=audit_events)

print("Input XML:")
print(demo_xml)
print("Extracted text (before cleaning):")
print(repr(raw_text))
print("Suppression events:")
for ev in audit_events:
    print(f"  <{ev['element']}> suppressed: {ev['suppressed']!r}  "
          f"(tail kept: {ev['tail_kept']!r})")

**Cleaning the extracted text**

After extraction, each chapter string goes through `_clean()` in three passes.

Pass 1 is Unicode NFC normalisation. The Perseus TEI files mix precomposed and decomposed Unicode for Latin characters with diacritics (macrons, etc.) -- `ā` might be `\u0101` (precomposed) or `a` + `\u0304` (decomposed combining macron). NFC normalisation converts everything to canonical precomposed form.

Pass 2 is pipe removal. The Perseus TEI files use `|` as an overline delimiter inside `<num>` elements for sestertius amounts (e.g. `|XX̄|` for two hundred sestertii). Since `|` is the CSV delimiter I've settled on (and stand by, at least so long as we are storing data in CSV format), any pipe surviving into the text column would corrupt the row. I replace pipes with a single space rather than deleting them outright, to avoid accidentally merging adjacent tokens: `"HS |X̅X̅,"` becomes `"HS  X̅X̅,"` (two spaces, collapsed in pass 3) rather than `"HSX̅X̅,"`.

Pass 3 collapses all whitespace runs to a single space and strips leading/trailng whitespace. XML pretty-printing inserts newlines and indentation that don't belong to the prose.

What I deliberately don't touch: Latin spelling variants (`u`/`v`, `i`/`j`, archaic forms), punctuation, and macrons. The downstream tagger will handle spelling variants, and stripping macrons would be a lossy transformation that's a hassle to try to undo later. The principle here is to make as few binding design decisions at this stage as possible.

In [ ]:
def _clean(text, _audit_pipes=None):
    """
    Three-pass cleaning: NFC normalisation, pipe removal, whitespace collapse.
    If _audit_pipes is a list, pipe-removal events get appended to it.
    """
    # Pass 1: NFC
    text_nfc = unicodedata.normalize("NFC", text)

    # Pass 2: pipe removal
    if "|" in text_nfc:
        count   = text_nfc.count("|")
        context = text_nfc[:200]
        if _audit_pipes is not None:
            _audit_pipes.append({"pipe_count": count, "context": context})
        text_pipes = text_nfc.replace("|", " ")
    else:
        text_pipes = text_nfc

    # Pass 3: whitespace collapse
    return re.sub(r"\s+", " ", text_pipes).strip()


# Demo -- one example per pass
demo_samples = [
    ("NFC normalisation",
     "a\u0304mo\u0304 ca\u0304rum  (decomposed macrons)",
     "decomposed macrons -> precomposed"),
    ("Pipe removal",
     "redegit HS |CCCL| in aerarium",
     "sestertius notation collides with CSV delimiter"),
    ("Whitespace collapse",
     "  Gallia est\n  omnis divisa\n  in partes tres  ",
     "newlines from XML indentation"),
]

pipe_audit = []
for title, raw, note in demo_samples:
    cleaned = _clean(raw, _audit_pipes=pipe_audit)
    print(f"[{title}]")
    print(f"  note   : {note}")
    print(f"  before : {raw!r}")
    print(f"  after  : {cleaned!r}")
    print()

**Parser**

Every Perseus TEI file wraps its text content in a `<div type="edition">`. I use that as the parsing root rather than the document root (`<TEI>`) because the `<teiHeader>` also contains text I don't want.

`_iter_divs()` finds `<div type="textpart" subtype="X">` elements within a parent, recursing through non-textpart wrappers but not into textpart divs of a different subtype.

In [ ]:
def _find_edition(root):
    """Find the top-level <div type='edition'> element."""
    for elem in root.iter(f"{{{TEI_NS}}}div"):
        if elem.get("type") == "edition":
            return elem
    raise ValueError(
        "No <div type='edition'> found. Is this a valid Perseus TEI file?"
    )


def _iter_divs(parent, subtype):
    """
    Yield <div type='textpart' subtype=subtype> elements.
    Recurses through non-textpart wrappers but not into
    textpart divs of a different subtype.
    """
    for child in parent:
        local = _local(child.tag)
        if local != "div":
            continue
        if (child.get("type") == "textpart"
                and child.get("subtype") == subtype):
            yield child
        elif child.get("type") != "textpart":
            yield from _iter_divs(child, subtype)


def parse_tei(xml_path, commentary, has_books, cts_base,
              pipe_audit=None, suppression_audit=None):
    """
    Parse a Perseus TEI file and return a list of row dicts.
    Each dict has keys: commentary, book, text, src, chapter.
    pipe_audit and suppression_audit are optional lists that
    collect transformation events for the audit log.
    """
    log.info(f"Parsing {xml_path.name} ({commentary}) ...")

    try:
        tree = ET.parse(xml_path)
    except ET.ParseError as e:
        log.error(f"XML parse error: {e}")
        return []

    root = tree.getroot()

    try:
        edition = _find_edition(root)
    except ValueError as e:
        log.error(str(e))
        return []

    rows = []

    if has_books:
        # lat2: book > chapter > (sections concatenated)
        for book_div in _iter_divs(edition, "book"):
            book_n = book_div.get("n", "0")
            try:
                book_num = int(book_n)
            except ValueError:
                log.warning(f"  Non-integer book n='{book_n}'; skipping")
                continue

            chapter_idx = 0
            for chap_div in _iter_divs(book_div, "chapter"):
                sa = []
                pa = []
                raw  = _extract_text(chap_div, _audit=sa)
                text = _clean(raw, _audit_pipes=pa)

                if suppression_audit is not None:
                    suppression_audit.extend(
                        dict(e, commentary=commentary, book=book_num,
                             chapter=chapter_idx+1)
                        for e in sa
                    )
                if pipe_audit is not None:
                    pipe_audit.extend(
                        dict(e, commentary=commentary, book=book_num,
                             chapter=chapter_idx+1)
                        for e in pa
                    )

                if not text:
                    log.debug(f"  Empty chapter: {commentary} bk{book_num} ch{chapter_idx+1}")
                    continue

                chap_n = chap_div.get("n", str(chapter_idx + 1))
                rows.append({
                    "commentary": commentary,
                    "book":       book_num,
                    "text":       text,
                    "src":        f"{cts_base}:{book_num}.{chap_n}",
                    "chapter":    chapter_idx,
                })
                chapter_idx += 1

    else:
        # lat1: chapter only, book always 1!
        chapter_idx = 0
        for chap_div in _iter_divs(edition, "chapter"):
            sa = []
            pa = []
            raw  = _extract_text(chap_div, _audit=sa)
            text = _clean(raw, _audit_pipes=pa)

            if suppression_audit is not None:
                suppression_audit.extend(
                    dict(e, commentary=commentary, book=1,
                         chapter=chapter_idx+1)
                    for e in sa
                )
            if pipe_audit is not None:
                pipe_audit.extend(
                    dict(e, commentary=commentary, book=1,
                         chapter=chapter_idx+1)
                    for e in pa
                )

            if not text:
                log.debug(f"  Empty chapter: {commentary} ch{chapter_idx+1}")
                continue

            chap_n = chap_div.get("n", str(chapter_idx + 1))
            rows.append({
                "commentary": commentary,
                "book":       1,
                "text":       text,
                "src":        f"{cts_base}:{chap_n}",
                "chapter":    chapter_idx,
            })
            chapter_idx += 1

    log.info(f"  Done: {len(rows)} chapters.")
    return rows

**Parse all five texts**

Suppression and pipe-removal events are collected as the parse runs, then reviewed in the audit cells below. This way we know what transformations have been done.

In [ ]:
global_pipe_audit        = []
global_suppression_audit = []
parse_stats              = {}
all_rows                 = []

for entry in CORPUS:
    commentary = entry["commentary"]
    if commentary not in xml_paths:
        log.warning(f"No XML file resolved for '{commentary}' -- skipping.")
        continue

    rows = parse_tei(
        xml_path          = xml_paths[commentary],
        commentary        = commentary,
        has_books         = entry["has_books"],
        cts_base          = entry["cts_base"],
        pipe_audit        = global_pipe_audit,
        suppression_audit = global_suppression_audit,
    )

    book_counts = defaultdict(int)
    for r in rows:
        book_counts[r["book"]] += 1
    parse_stats[commentary] = dict(book_counts)

    all_rows.extend(rows)

print(f"Total rows: {len(all_rows)}")

**Parse statistics**

In [ ]:
print(f"{'text':>12}  {'book':>4}  {'chapters':>8}  {'avg words/ch':>14}")
print("-" * 50)

for commentary in ["gallic", "civil", "alexandrine", "african", "spanish"]:
    if commentary not in parse_stats:
        print(f"{commentary:>12}  (no data)")
        continue
    for book_num in sorted(parse_stats[commentary]):
        book_rows = [
            r for r in all_rows
            if r["commentary"] == commentary and r["book"] == book_num
        ]
        n_ch    = len(book_rows)
        n_wds   = sum(len(r["text"].split()) for r in book_rows)
        avg_wds = n_wds / n_ch if n_ch else 0
        print(f"{commentary:>12}  {book_num:>4}  {n_ch:>8}  {avg_wds:>14.1f}")

**Spot checks**

First and last chapter of each text/book to verify the parse started and ended in the right place and that no markup leaked into the prose.

In [ ]:
def show_sample(rows, commentary, book=1, preview_len=200):
    """Print the first and last chapter of a given text/book."""
    book_rows = [
        r for r in rows
        if r["commentary"] == commentary and r["book"] == book
    ]
    if not book_rows:
        print(f"  No rows for {commentary} bk{book}")
        return

    for idx, label in [(0, "first"), (-1, "last")]:
        r       = book_rows[idx]
        preview = r["text"][:preview_len]
        if len(r["text"]) > preview_len:
            preview += "..."
        print(f"  [{label} | ch_idx={r['chapter']} | {r['src']}]")
        print(f"  {preview}")
        print()


for commentary, book in [
    ("gallic", 1), ("gallic", 7), ("gallic", 8),
    ("civil",  1), ("civil",  3),
    ("alexandrine", 1), ("african", 1), ("spanish", 1),
]:
    if commentary not in parse_stats:
        continue
    print(f"\n{commentary.upper()} book {book}")
    print("-" * 40)
    show_sample(all_rows, commentary, book=book)

**Suppression audit**

Every time the parser hit a `<note>`, `<del>`, `<head>`, etc. with non-empty text content, it recorded what was dropped. This table is the complete record of editorial content removed from the corpus. Worth reviewing to confirm nothing was suppressed that should have been kept.

In [ ]:
print(f"Total suppression events: {len(global_suppression_audit)}")

if global_suppression_audit:
    by_elem = Counter(ev["element"] for ev in global_suppression_audit)
    print("\nBy element type:")
    for elem, count in sorted(by_elem.items(), key=lambda x: -x[1]):
        print(f"  <{elem}> : {count}")

    print("\nSample (first 20 events):")
    print(f"  {'text':>12}  {'bk':>3}  {'ch':>3}  {'elem':>10}  suppressed (preview)")
    print("-" * 90)
    for ev in global_suppression_audit[:20]:
        print(f"  {ev.get('commentary','?'):>12}  "
              f"{ev.get('book','?'):>3}  "
              f"{ev.get('chapter','?'):>3}  "
              f"<{ev['element']:>8}>  "
              f"{ev['suppressed']!r:.60}")
else:
    print("  (no suppression events)")

**Pipe-removal audit**

Every chapter where the `|` to space substitution was applied. Pipes in the Perseus TEI typically come from sestertius notation inside `<num>` elements.

In [ ]:
print(f"Total pipe-removal events: {len(global_pipe_audit)}")

if global_pipe_audit:
    print(f"\n  {'text':>12}  {'bk':>3}  {'ch':>3}  {'pipes':>5}  context")
    print("-" * 80)
    for ev in global_pipe_audit:
        ctx = ev["context"][:50].replace("\n", " ")
        print(f"  {ev.get('commentary','?'):>12}  "
              f"{ev.get('book','?'):>3}  "
              f"{ev.get('chapter','?'):>3}  "
              f"{ev['pipe_count']:>5}  {ctx!r}")
else:
    print("  (no pipe characters found)")

**NFC normalisation -- impact check**

In [ ]:
nfc_changed  = 0
nfc_examples = []

for r in all_rows:
    text = r["text"]
    if unicodedata.normalize("NFD", text) != text:
        nfc_changed += 1
        if len(nfc_examples) < 3:
            for char in text:
                if ord(char) > 127:
                    nfc_examples.append({
                        "src":  r["src"],
                        "char": char,
                        "cp":   f"U+{ord(char):04X}",
                        "name": unicodedata.name(char, "?"),
                    })
                    break

print(f"Chapters with non-ASCII characters (post-NFC): {nfc_changed}")
if nfc_examples:
    print("\nExamples:")
    for ex in nfc_examples:
        print(f"  {ex['src']:50}  {ex['char']!r}  {ex['cp']}  {ex['name']}")
else:
    print("  (all text is ASCII)")

**Chapter-count validation**

Cross-check actual chapter counts against standard scholarly editions (OCT/Teubner). A mismatch most likely means a structural parsing problem, an incomplete XML file, or a divergence between the Perseus editorial conventions and the standard editions. See the notes at the end of this notebook for known issues.

In [ ]:
EXPECTED_CHAPTERS = {
    ("gallic", 1): 54,  ("gallic", 2): 35,  ("gallic", 3): 29,
    ("gallic", 4): 38,  ("gallic", 5): 58,  ("gallic", 6): 44,
    ("gallic", 7): 90,  ("gallic", 8): 56,
    ("civil",  1): 87,  ("civil",  2): 44,  ("civil",  3): 112,
    ("alexandrine", 1): 78,
    ("african",     1): 98,
    ("spanish",     1): 42,
}


def validate(rows, expected=EXPECTED_CHAPTERS):
    counts = Counter((r["commentary"], r["book"]) for r in rows)
    all_ok = True

    print(f"  {'text/book':25}  {'got':>5}  {'expected':>8}  status")
    print("-" * 55)

    for key in sorted(expected):
        commentary, book = key
        exp    = expected[key]
        got    = counts.get(key, 0)
        label  = f"{commentary} bk{book}"
        status = "ok" if got == exp else f"MISMATCH (diff {got-exp:+d})"
        if got != exp:
            all_ok = False
        print(f"  {label:25}  {got:>5}  {exp:>8}  {status}")

    print()
    if all_ok:
        print("  All chapter counts match.")
    else:
        print("  Some chapter counts don't match. See notes at end of notebook.")
    return all_ok


validation_passed = validate(all_rows)

**Write the CSV**

The format matches `full_data_text.csv` exactly -- pipe-delimited, UTF-8, integer index as the first column, same column order -- so it's a drop-in replacement for the analysis pipeline.

In [ ]:
def write_csv(rows, out_path):
    """Write rows to a pipe-delimited CSV matching full_data_text.csv format."""
    columns = ["commentary", "book", "text", "src", "chapter"]
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        f.write("|" + "|".join(columns) + "\n")
        for i, row in enumerate(rows):
            if "|" in row["text"]:
                log.error(
                    f"  Pipe survived cleaning at row {i} ({row['src']}) -- "
                    "row will be corrupted!"
                )
            fields = [
                str(i),
                row["commentary"],
                str(row["book"]),
                row["text"],
                row["src"],
                str(row["chapter"]),
            ]
            f.write("|".join(fields) + "\n")

    size_kb = Path(out_path).stat().st_size // 1024
    log.info(f"Written {len(rows):,} rows ({size_kb:,} KB) to {out_path}")


write_csv(all_rows, OUTPUT_PATH)

**Round-trip check**

Read the CSV back and verify column count on every row. A miscount means a pipe survived into the text column.

In [ ]:
with open(OUTPUT_PATH, encoding="utf-8") as f:
    lines = f.readlines()

header     = lines[0].rstrip("\n").split("|")
data_lines = lines[1:]

print(f"Header columns : {header}")
print(f"Data rows      : {len(data_lines):,}  (expected {len(all_rows):,})")

errors = []
for lineno, line in enumerate(data_lines, start=2):
    parts = line.rstrip("\n").split("|")
    if len(parts) != len(header):
        errors.append(
            f"  Line {lineno}: expected {len(header)} fields, got {len(parts)}"
        )

if errors:
    print(f"\n{len(errors)} malformed rows:")
    for e in errors[:10]:
        print(e)
else:
    print("All rows have the correct number of fields. Round-trip check passed.")

**Final stats**

In [ ]:
print("Corpus Caesarianum -- final corpus statistics")
print("=" * 60)
print(f"  {'text':>12}  {'book':>4}  {'ch.':>5}  "
      f"{'words':>8}  {'avg w/ch':>9}  {'min':>5}  {'max':>5}")
print("-" * 62)

grand_ch  = 0
grand_wds = 0

for commentary in ["gallic", "civil", "alexandrine", "african", "spanish"]:
    if commentary not in parse_stats:
        continue
    for book_num in sorted(parse_stats[commentary]):
        book_rows = [
            r for r in all_rows
            if r["commentary"] == commentary and r["book"] == book_num
        ]
        wc = [len(r["text"].split()) for r in book_rows]
        n  = len(wc)
        s  = sum(wc)
        grand_ch  += n
        grand_wds += s
        print(f"  {commentary:>12}  {book_num:>4}  {n:>5}  "
              f"{s:>8,}  {s/n if n else 0:>9.1f}  "
              f"{min(wc) if wc else 0:>5}  {max(wc) if wc else 0:>5}")

print("-" * 62)
print(f"  {'total':>12}  {'':>4}  {grand_ch:>5}  {grand_wds:>8,}")
print("=" * 60)

**Transformation audit summary**

Everything that was changed from the source data in one place.

In [ ]:
print("Transformation audit summary")
print("=" * 60)

print("\nFile acquisition")
for commentary, method, path, size_kb in acquisition_log:
    print(f"  {commentary:>12}  [{method:>8}]  {size_kb:>5} KB  {path}")

print(f"\nElement suppression ({len(global_suppression_audit)} events)")
by_elem = Counter(ev["element"] for ev in global_suppression_audit)
for elem, count in sorted(by_elem.items(), key=lambda x: -x[1]):
    print(f"  <{elem}> : {count} instances suppressed")
if not global_suppression_audit:
    print("  (none)")

print(f"\nPipe removal ({len(global_pipe_audit)} chapters affected)")
if global_pipe_audit:
    total_pipes = sum(ev["pipe_count"] for ev in global_pipe_audit)
    print(f"  {total_pipes} pipe characters replaced with spaces")
else:
    print("  (none)")

print(f"\nUnicode normalisation")
print(f"  NFC applied to all {len(all_rows):,} chapters")

print(f"\nWhitespace normalisation")
print(f"  Whitespace collapsed across all {len(all_rows):,} chapters")

print(f"\nValidation")
print(f"  Chapter-count check: {'passed' if validation_passed else 'FAILED -- see notes below'}")

print(f"\nOutput")
print(f"  File   : {OUTPUT_PATH}")
print(f"  Rows   : {len(all_rows):,}")
print(f"  Format : pipe-delimited UTF-8 CSV")
print("=" * 60)

**Loading the output with pandas**

```python
import pandas as pd

df = pd.read_csv(
    "full_data_text_perseus.csv",
    sep="|",
    index_col=0,
    dtype={"book": int, "chapter": int},
)

print(df.shape)
print(df["commentary"].value_counts())
print(df.head())
```

**Known divergences from standard editions**

*Bellum Civile.* The Perseus edition follows the Klotz (1926) numbering. Alternative numberings (e.g. Damon 2015 for Loeb) split or combine a small number of chapters differently. If the validation check flags a mismatch in *BC*, that's the likely source.

*De Bello Gallico*, book 8. The preface (*praefatio*) addressed by Hirtius to Balbus is included by some editions as an unnumbered prologue and excluded by others. The Perseus edition includes it as part of the text but doesn't break it out as a separate chapter -- so it gets incorporated into the first chapter's text. Chapter 0 of `gallic` book 8 may be longer than expected because of this.

*Bellum Hispaniense.* The most textually corrupt of the five works. Several chapters contain `<gap>` elements marking lacunae. Since `<gap>` content is suppressed per the extraction rules, chapters with significant gaps will have shorter text than the standard editions suggest. The word-count stats above will reflect this.